In [1]:
import os
import pandas as pd
import numpy as np
import datetime as dt
from tqdm import tqdm

In [2]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

Latest run date: 2025-03-10 20:44:04.834346


#### Functions

#### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

Project: 20250307-funded-trends
Task: 04_join_targets


#### Output dir

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [5]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/01_data_collection/{str_filename}'
df = pd.read_parquet(str_uri)
df['accountid'] = df['accountid'].astype(int)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,SUPER.COM_tag,STEP MOBILE_tag,BRIGHT_tag,FIG TECH INC_tag,SELFBILLSE_tag,PROGRESSRES_tag,FLEX_tag,FLEXFINANCE_tag,sum,has_inst_tag
1,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0,0,0,0,0,0,0,0,0,0
2,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0,0,0,0,0,0,0,0,0,0
0,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0,0,0,0,0,0,0,0,0,0
42,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0,0,0,0,0,0,0,0,0,0
41,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0,0,0,0,0,0,0,0,1,1
3836,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0,0,0,0,0,0,0,0,0,0
3763,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0,0,0,0,0,0,0,0,0,0
3702,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0,0,0,0,0,0,0,0,3,1


#### Drop targets

In [6]:
list_cols = [col for col in df.columns if 'Early_Pay_Delinquency' in col]
list_cols.append('run_date')
df.drop(list_cols, axis=1, inplace=True)

#### Get targets - classification

In [7]:
str_filename = 'df_targets.gzip'
str_uri = f's3://{str_project}/03_pull_targets/01_classification/{str_filename}'
df_tmp = pd.read_parquet(str_uri)
df_tmp['bigAccountId'] = df_tmp['bigAccountId'].astype(int)
dict_rename = {
    'bigAccountId': 'accountid',
}
df_tmp.rename(columns=dict_rename, inplace=True)
df_tmp

,accountid,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date
0,6,0,0,0,0,0,2025-03-10 14:38:45.066326
1,25,1,0,0,1,1,2025-03-10 14:38:45.066326
2,73,0,0,0,0,1,2025-03-10 14:38:45.066326
3,82,1,1,1,1,1,2025-03-10 14:38:45.066326
4,122,0,0,0,1,1,2025-03-10 14:38:45.066326
...,...,...,...,...,...,...,...
344643,8713064,0,0,0,0,0,2025-03-10 14:38:45.066326
344644,8713405,0,0,0,0,0,2025-03-10 14:38:45.066326
344645,8715756,0,0,0,0,0,2025-03-10 14:38:45.066326
344646,8716150,0,0,0,0,0,2025-03-10 14:38:45.066326


#### Join

In [8]:
df = pd.merge(
    left=df,
    right=df_tmp,
    on='accountid',
    how='left',
)
del df_tmp
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,FLEX_tag,FLEXFINANCE_tag,sum,has_inst_tag,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date
0,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0,0,0,0,1,0,1,1,1,2025-03-10 14:38:45.066326
1,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0,0,0,0,0,0,0,0,0,2025-03-10 14:38:45.066326
2,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0,0,0,0,0,0,0,0,0,2025-03-10 14:38:45.066326
3,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0,0,0,0,0,0,1,1,0,2025-03-10 14:38:45.066326
4,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0,0,0,0,0,0,1,1,0,2025-03-10 14:38:45.066326
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0,0,1,1,0,0,0,0,0,2025-03-10 14:38:45.066326
100027,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0,0,0,0,0,0,0,0,0,2025-03-10 14:38:45.066326
100028,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0,0,0,0,0,0,0,0,0,2025-03-10 14:38:45.066326
100029,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0,0,3,1,0,0,0,0,0,2025-03-10 14:38:45.066326


#### Get targets - continuous

In [9]:
str_filename = 'df_loss.gzip'
str_uri = f's3://{str_project}/03_pull_targets/02_regression/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
    columns=['bigAccountId','fltNetChgOff','months_on_books'],
)
df_tmp['bigAccountId'] = df_tmp['bigAccountId'].astype(int)
dict_rename = {
    'bigAccountId': 'accountid',
}
df_tmp.rename(columns=dict_rename, inplace=True)
# show
df_tmp

,accountid,fltNetChgOff,months_on_books
index,,,
0,370217,6796.08,160
1,306072,17702.55,170
2,245270,4755.88,182
3,196070,9161.37,209
4,239071,15970.30,184
...,...,...,...
142926,5673846,880.25,44
142927,5793530,6427.40,38
142928,5927894,252.37,36


#### Join

In [10]:
list_int_months = [
    2,
    3,
    6,
    12,
    24,
]
for int_months in tqdm(list_int_months):
    df_tmp2 = df_tmp[df_tmp['months_on_books'] == int_months].copy()
    dict_rename = {
        'fltNetChgOff': f'fltNetChgOff_{int_months}',
    }
    df_tmp2.rename(columns=dict_rename, inplace=True)
    # join
    list_cols = [
        'accountid',
        f'fltNetChgOff_{int_months}',
    ]
    df = pd.merge(
        left=df,
        right=df_tmp2[list_cols],
        on='accountid',
        how='left',
    )
# save memory
del df_tmp
del df_tmp2

# show
df

100%|██████████| 5/5 [00:09<00:00,  1.96s/it]


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date,fltNetChgOff_2,fltNetChgOff_3,fltNetChgOff_6,fltNetChgOff_12,fltNetChgOff_24
0,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0,1,1,1,2025-03-10 14:38:45.066326,NaN,NaN,NaN,NaN,7827.16
1,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0,0,0,0,2025-03-10 14:38:45.066326,NaN,NaN,NaN,NaN,NaN
2,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0,0,0,0,2025-03-10 14:38:45.066326,NaN,NaN,NaN,NaN,NaN
3,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0,1,1,0,2025-03-10 14:38:45.066326,NaN,NaN,NaN,NaN,NaN
4,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0,1,1,0,2025-03-10 14:38:45.066326,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0,0,0,0,2025-03-10 14:38:45.066326,NaN,NaN,NaN,NaN,NaN
100027,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0,0,0,0,2025-03-10 14:38:45.066326,NaN,NaN,NaN,NaN,NaN
100028,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0,0,0,0,2025-03-10 14:38:45.066326,NaN,NaN,NaN,NaN,NaN
100029,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0,0,0,0,2025-03-10 14:38:45.066326,NaN,NaN,NaN,NaN,NaN


#### Fillna

In [11]:
list_cols = [col for col in df.columns if 'fltNetChgOff' in col]
df[list_cols] = df[list_cols].fillna(0)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date,fltNetChgOff_2,fltNetChgOff_3,fltNetChgOff_6,fltNetChgOff_12,fltNetChgOff_24
0,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0,1,1,1,2025-03-10 14:38:45.066326,0.0,0.0,0.0,0.0,7827.16
1,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0,0,0,0,2025-03-10 14:38:45.066326,0.0,0.0,0.0,0.0,0.00
2,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0,0,0,0,2025-03-10 14:38:45.066326,0.0,0.0,0.0,0.0,0.00
3,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0,1,1,0,2025-03-10 14:38:45.066326,0.0,0.0,0.0,0.0,0.00
4,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0,1,1,0,2025-03-10 14:38:45.066326,0.0,0.0,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0,0,0,0,2025-03-10 14:38:45.066326,0.0,0.0,0.0,0.0,0.00
100027,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0,0,0,0,2025-03-10 14:38:45.066326,0.0,0.0,0.0,0.0,0.00
100028,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0,0,0,0,2025-03-10 14:38:45.066326,0.0,0.0,0.0,0.0,0.00
100029,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0,0,0,0,2025-03-10 14:38:45.066326,0.0,0.0,0.0,0.0,0.00


#### Charge-off severity

In [12]:
# 60 days
df['co_at_60'] = df['fltNetChgOff_2'] / df['amtfinanced__app']
# 90 days
df['co_at_90'] = df['fltNetChgOff_3'] / df['amtfinanced__app']
# 180
df['co_at_180'] = df['fltNetChgOff_6'] / df['amtfinanced__app']
# 360
df['co_at_360'] = df['fltNetChgOff_12'] / df['amtfinanced__app']
# 720
df['co_at_720'] = df['fltNetChgOff_24'] / df['amtfinanced__app']
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,fltNetChgOff_2,fltNetChgOff_3,fltNetChgOff_6,fltNetChgOff_12,fltNetChgOff_24,co_at_60,co_at_90,co_at_180,co_at_360,co_at_720
0,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0.0,0.0,0.0,0.0,7827.16,0.0,0.0,0.0,0.0,0.315976
1,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
2,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
3,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
4,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100027,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100028,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100029,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000


#### Write to s3

In [13]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)

CPU times: user 34.9 s, sys: 244 ms, total: 35.1 s
Wall time: 35.1 s
